# CHUYÊN ĐỀ ĐỈNH CAO: BIẾN CHIA SẺ PHÂN TÁN (SHARED VARIABLES) TRÊN MICROSOFT FABRIC
## BROADCAST VARIABLES & ACCUMULATORS – KỸ THUẬT TIẾT KIỆM BĂNG THÔNG MẠNG & AUDIT DATA QUALITY

---

### 1. Vấn đề cốt lõi trong tính toán phân tán (The Closure Problem):
Trong mô hình cụm Spark (Cluster Architecture):
- **Driver:** Chạy mã điều phối chính, nắm giữ biến và logic truy vấn.
- **Executors (Worker Nodes):** Các máy chủ tính toán dữ liệu song song.

Khi bạn khai báo một biến Python thông thường (như một `dict` danh mục 50MB) và truyền vào hàm tính toán song song:
- Spark buộc phải **serialize (đóng gói) và gửi biến này qua mạng tới TỪNG TASK một**!
- Nếu có 1,000 tasks, biến 50MB sẽ bị sao chép 1,000 lần $\rightarrow$ **Tốn 50 GB băng thông mạng nội bộ, gây tràn RAM và nghẽn hệ thống!**

---

### 2. Hai vũ khí giải cứu:
1. **Broadcast Variable (`sc.broadcast`):**
   - **Đặc tính:** CHỈ ĐỌC (Read-Only).
   - **Cơ chế:** Driver chỉ gửi biến **1 LẦN DUY NHẤT** đến mỗi máy Executor (thay vì mỗi Task). Mọi task trên Executor đó dùng chung một bản sao trong RAM.
   - **Ứng dụng:** Danh mục tra cứu nhanh (lookup dict), bộ quy đổi tỷ giá, danh sách blacklist, trọng số Machine Learning...

2. **Accumulator (`sc.accumulator`):**
   - **Đặc tính:** CHỈ GHI (Write-Only từ Executor, Read-Only từ Driver).
   - **Cơ chế:** Các task phân tán chỉ có quyền **cộng dồn (+1, +value)** vào biến này lúc đang xử lý dữ liệu. Chỉ duy nhất Driver mới có quyền đọc giá trị tổng kết.
   - **Ứng dụng:** Đếm số dòng lỗi, audit data quality, bắt ngoại lệ **trong đúng 1 lần quét dữ liệu (Single Pass)** mà không cần quét lại đĩa!

## PHẦN 1: KHỞI TẠO MÔI TRƯỜNG & LẤY SPARKCONTEXT TRÊN FABRIC

Trên Microsoft Fabric:
- `spark`: Là đối tượng **SparkSession** (xử lý DataFrame, SQL, Delta Lake).
- `sc`: Là đối tượng **SparkContext** (quản lý RDD, kết nối phần cứng và biến chia sẻ phân tán).
Ta có thể truy xuất `sc` trực tiếp hoặc qua `spark.sparkContext`.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

# Lấy SparkContext từ SparkSession trên Fabric
sc = spark.sparkContext

print(f"=== KẾT NỐI SPARK CONTEXT THÀNH CÔNG ===")
print(f"- Spark Version: {spark.version}")
print(f"- Application ID: {sc.applicationId}")
print(f"- Default Parallelism: {sc.defaultParallelism}")


## PHẦN 2: SO SÁNH THỰC TẾ: KHÔNG DÙNG BROADCAST VS CÓ DÙNG BROADCAST VARIABLE

**Bài toán thực tế:**
Bạn có 1,000,000 giao dịch logistics quốc tế mang nhiều loại ngoại tệ (`USD`, `EUR`, `JPY`, `GBP`, `SGD`, `VND`).
Bạn cần quy đổi tất cả sang VNĐ bằng một bảng tỷ giá (`exchange_rate_dict`) và kiểm tra danh sách khách hàng gian lận (`fraud_blacklist_set`).

---

### 🔍 Cơ chế hoạt động dưới nền (Under the Hood):

1. **Khi KHÔNG dùng Broadcast Variable (Closure Capture):**
   - Mỗi khi Spark tạo Task, nó sẽ đóng gói (Serialize / Pickle) hàm UDF cùng toàn bộ biến `exchange_rate_dict` và `fraud_blacklist_set` nằm trong closure.
   - Nếu dữ liệu chia làm **40 Partitions (40 Tasks)**, đối tượng Python Dict/Set sẽ bị sao chép và **gửi 40 lần qua mạng nội bộ**!
   - **Hậu quả:** Tốn thời gian Serialization/Deserialization, gây nghẽn băng thông mạng (Network I/O) và lãng phí RAM của Executor để chứa 40 bản sao trùng lặp.

2. **Khi CÓ dùng Broadcast Variable (`sc.broadcast`):**
   - Spark Driver đóng gói dữ liệu và gửi **1 LẦN DUY NHẤT** đến bộ nhớ RAM của mỗi Executor (Worker Node).
   - Tất cả 40 Tasks chạy trên Executor đó đều truy cập chung một bản sao duy nhất trong RAM thông qua thuộc tính `.value`.
   - **Kết quả:** Tiết kiệm tối đa băng thông mạng nội bộ, giảm latency khi khởi chạy Task và tối ưu hóa bộ nhớ RAM.

In [ ]:
# 1. Khởi tạo dữ liệu tra cứu tại Driver
exchange_rate_dict = {
    "USD": 25450.0,
    "EUR": 27300.0,
    "JPY": 168.0,
    "GBP": 32500.0,
    "SGD": 18900.0,
    "VND": 1.0
}
fraud_blacklist_set = {f"CUST_{i:03d}" for i in range(7, 1000, 10)}  # 100 gian lận viên

# 2. Tạo DataFrame 1,000,000 giao dịch phân thành 40 Partitions (40 Tasks thực thi)
currencies = ["USD", "EUR", "JPY", "GBP", "SGD", "VND"]
df_orders = spark.range(0, 1000000) \
    .repartition(40) \
    .withColumn("order_id", F.concat(F.lit("ORD_"), F.col("id"))) \
    .withColumn("cust_id", F.concat(F.lit("CUST_"), F.format_string("%03d", (F.col("id") % 1000)))) \
    .withColumn("currency", F.elt(F.col("id") % 6 + 1, *[F.lit(c) for c in currencies])) \
    .withColumn("raw_amount", F.round(F.rand() * 1000 + 10, 2)) \
    .drop("id")

# ==============================================================================
# CÁCH 1: KHÔNG DÙNG BROADCAST VARIABLE (TRUYỀN TRỰC TIẾP BIẾN PYTHON VÀO CLOSURE)
# ==============================================================================
print("=== [TEST 1] THỰC THI KHÔNG DÙNG BROADCAST VARIABLE ===")

@F.udf(returnType=DoubleType())
def convert_no_bc_udf(curr, amount):
    # Truy cập trực tiếp biến exchange_rate_dict từ closure của Python (Bị copy theo từng Task!)
    rate = exchange_rate_dict.get(curr, 1.0)
    return float(amount * rate)

@F.udf(returnType=BooleanType())
def check_fraud_no_bc_udf(cust_id):
    # Truy cập trực tiếp biến fraud_blacklist_set từ closure của Python
    return cust_id in fraud_blacklist_set

t0 = time.time()
df_no_bc = df_orders \
    .withColumn("amount_vnd", convert_no_bc_udf(F.col("currency"), F.col("raw_amount"))) \
    .withColumn("is_fraud", check_fraud_no_bc_udf(F.col("cust_id")))

cnt1 = df_no_bc.count()
t_no_bc = time.time() - t0
print(f"-> Hoàn thành {cnt1:,} dòng (Không dùng Broadcast) trong: {t_no_bc:.3f} giây")


# ==============================================================================
# CÁCH 2: CÓ DÙNG BROADCAST VARIABLE (sc.broadcast)
# ==============================================================================
print("\n=== [TEST 2] THỰC THI CÓ DÙNG BROADCAST VARIABLE ===")

# Broadcast biến từ Driver xuống tất cả các Executor (Chỉ gửi 1 lần duy nhất per Executor!)
b_exchange_rates = sc.broadcast(exchange_rate_dict)
b_fraud_blacklist = sc.broadcast(fraud_blacklist_set)

@F.udf(returnType=DoubleType())
def convert_with_bc_udf(curr, amount):
    # Đọc từ Broadcast Variable qua thuộc tính .value (Dùng chung bộ nhớ RAM của Executor)
    rate = b_exchange_rates.value.get(curr, 1.0)
    return float(amount * rate)

@F.udf(returnType=BooleanType())
def check_fraud_with_bc_udf(cust_id):
    # Đọc từ Broadcast Variable qua thuộc tính .value
    return cust_id in b_fraud_blacklist.value

t0 = time.time()
df_with_bc = df_orders \
    .withColumn("amount_vnd", convert_with_bc_udf(F.col("currency"), F.col("raw_amount"))) \
    .withColumn("is_fraud", check_fraud_with_bc_udf(F.col("cust_id")))

cnt2 = df_with_bc.count()
t_bc = time.time() - t0
print(f"-> Hoàn thành {cnt2:,} dòng (Có dùng Broadcast) trong: {t_bc:.3f} giây")


# ==============================================================================
# BÁO CÁO THỐNG KÊ VÀ SO SÁNH NGUYÊN LÝ
# ==============================================================================
print("\n=======================================================================")
print("=== BẢNG SO SÁNH NGUYÊN LÝ & HIỆU NĂNG TẢI NẠNG MẠNG ===")
print("=======================================================================")
num_tasks = df_orders.rdd.getNumPartitions()
print(f"Số lượng Task (Partitions): {num_tasks}")
print(f"- Khi KHÔNG dùng Broadcast : Biến Dict/Set bị serialize và gửi {num_tasks} lần qua mạng.")
print(f"- Khi CÓ dùng Broadcast    : Biến Dict/Set chỉ gửi 1 lần duy nhất per Executor.")
print(f"- Thời gian chạy Không Broadcast : {t_no_bc:.3f}s")
print(f"- Thời gian chạy Có Broadcast    : {t_bc:.3f}s")
if t_no_bc > 0:
    diff_pct = ((t_no_bc - t_bc) / t_no_bc) * 100
    print(f"-> Broadcast Variable giúp tối ưu băng thông mạng & tăng tốc khoảng: {diff_pct:.1f}%")
print("=======================================================================")

print("\n=== MẪU DỮ LIỆU SAU KHI TRA CỨU BẰNG BROADCAST VARIABLE ===")
df_with_bc.filter("is_fraud = TRUE").show(5, truncate=False)


## PHẦN 3: SO SÁNH 4 PHƯƠNG PHÁP AUDIT DATA QUALITY (ACCUMULATOR VS PYTHON VARIABLE VS MULTI-PASS VS DATAFRAME NATIVE)

**Bài toán Data Quality Audit trong Logistics:**
Bạn nạp 500,000 bản ghi dữ liệu vận tải thô (`raw_logistics_data`). Bạn cần vừa làm sạch dữ liệu (Clean Data), vừa **đếm số bản ghi bị vi phạm các lỗi nghiệp vụ**:
- Lỗi 1: Format Email sai / thiếu.
- Lỗi 2: Cước phí vận chuyển bị âm (`shipping_fee < 0`).
- Lỗi 3: Thiếu mã bưu cục (`pos_code` là NULL hoặc rỗng).

---

### 🔍 Sự khác biệt về nguyên lý vận hành giữa 4 phương pháp:

1. **❌ Cách 1: Dùng biến Python thông thường (`py_counter = 0`) trong hàm phân tán**
   - Biến Python `py_counter = 0` được tạo ở Driver. Khi Executor chạy hàm `map/filter`, nó tăng giá trị trên bản sao local của Executor đó.
   - **Thất bại hoàn toàn:** Các Executor **không có cơ chế gửi lại biến Python thường về Driver**. Sau khi Job hoàn thành, ở Driver biến `py_counter` **VẪN BẰNG 0**!

2. **⚠️ Cách 2: Chạy nhiều lệnh `.filter().count()` riêng biệt (Multi-Pass)**
   - Đọc dữ liệu $\rightarrow$ Lọc lỗi Email $\rightarrow$ `.count()` (Quét đĩa lần 1).
   - Đọc lại dữ liệu $\rightarrow$ Lọc lỗi Cước âm $\rightarrow$ `.count()` (Quét đĩa lần 2).
   - Đọc lại dữ liệu $\rightarrow$ Lọc lỗi POS $\rightarrow$ `.count()` (Quét đĩa lần 3).
   - **Hậu quả:** Spark phải đọc lại đĩa/bộ nhớ **4 lần liên tiếp**, làm tăng thời gian xử lý và tải đĩa.

3. **🐢 Cách 3: Dùng RDD Accumulator (`sc.accumulator(0)`) (Single-Pass RDD)**
   - Khởi tạo Accumulator tại Driver. Các Worker gọi `acc.add(1)` trong quá trình lọc. Spark tự động tổng hợp số đếm từ các Worker gửi về Driver.
   - **Ưu điểm:** Quét dữ liệu 1 lần duy nhất (Single-pass).
   - **Hạn chế:** Chạy qua Python RDD Filter làm tốn chi phí **Serialization/Deserialization (Python SerDe Overhead)** khi chuyển dữ liệu từ JVM sang Python Process.

4. **🚀 Cách 4: Dùng DataFrame Conditional Aggregation (`F.when()`) (Single-Pass Native Spark Engine - CHUẨN PRODUCTION)**
   - Sử dụng các biểu thức Spark SQL / DataFrame native (`F.count(F.when(...))`) để đếm lỗi trực tiếp trên DataFrame.
   - **Kết quả vượt trội:** Vừa **quét dữ liệu 1 lần duy nhất (Single-pass)**, vừa chạy **100% Native trong JVM (Catalyst / Tungsten Engine)** không tốn chi phí Python SerDe $\rightarrow$ Tốc độ siêu tốc!

In [ ]:
# 1. Tạo tập dữ liệu 500,000 dòng giả lập logistics chứa nhiều lỗi thực tế
import random

raw_data = []
for i in range(500000):
    c_id = f"C_{i:06d}"
    # Giả lập lỗi email (10% lỗi)
    email = f"user_{i}@logistics.vn" if i % 10 != 0 else "invalid_email_format"
    # Giả lập lỗi cước âm (5% lỗi)
    fee = float(random.randint(10, 500)) if i % 20 != 0 else -100.0
    # Giả lập lỗi thiếu POS code (8% lỗi)
    pos = f"POS_{(i % 5) + 1:02d}" if i % 12 != 0 else None
    raw_data.append((c_id, email, fee, pos))

df_raw = spark.createDataFrame(raw_data, ["cust_id", "email", "shipping_fee", "pos_code"]).repartition(20)


# ==============================================================================
# CÁCH 1: THỬ DÙNG BIẾN PYTHON THÔNG THƯỜNG (THẤT BẠI - BỊ NUỐT KẾT QUẢ)
# ==============================================================================
print("=== [TEST 1] KHÔNG DÙNG ACCUMULATOR: DÙNG BIẾN PYTHON THƯỜNG (py_counter = 0) ===")
py_invalid_email = 0
py_negative_amount = 0

def bad_validate_func(row):
    global py_invalid_email, py_negative_amount
    # Cố tình tăng biến Python thường ở Executor
    if row.email is None or "@" not in row.email:
        py_invalid_email += 1
    if row.shipping_fee is None or row.shipping_fee < 0:
        py_negative_amount += 1
    return True

t0 = time.time()
df_raw.rdd.filter(bad_validate_func).count()
t_py = time.time() - t0

print(f"-> Kết quả đọc được tại Driver sau khi chạy phân tán:")
print(f"   + py_invalid_email   = {py_invalid_email}  ❌ (Thất bại! Vẫn bằng 0 do Executor không gửi về được)")
print(f"   + py_negative_amount = {py_negative_amount}  ❌ (Thất bại! Vẫn bằng 0)")


# ==============================================================================
# CÁCH 2: GỌI MULTI-PASS FILTER().COUNT() N LẦN (TỐN THỜI GIAN QUÉT ĐĨA/LƯU TRỮ)
# ==============================================================================
print("\n=== [TEST 2] KHÔNG DÙNG ACCUMULATOR: GỌI MULTI-PASS FILTER().COUNT() ===")
t0 = time.time()

cnt_err_email = df_raw.filter("email IS NULL OR email NOT LIKE '%@%'").count()
cnt_err_fee   = df_raw.filter("shipping_fee IS NULL OR shipping_fee < 0").count()
cnt_err_pos   = df_raw.filter("pos_code IS NULL OR trim(pos_code) = ''").count()
df_clean_multi = df_raw.filter("email LIKE '%@%' AND shipping_fee >= 0 AND pos_code IS NOT NULL")
cnt_clean_multi = df_clean_multi.count()

t_multi = time.time() - t0
print(f"-> Hoàn thành Multi-Pass Audit (Quét dữ liệu 4 lần) trong: {t_multi:.3f} giây")
print(f"   + Lỗi Email    : {cnt_err_email:,} dòng")
print(f"   + Lỗi Phí âm   : {cnt_err_fee:,} dòng")
print(f"   + Lỗi Thiếu POS : {cnt_err_pos:,} dòng")
print(f"   + Dữ liệu sạch  : {cnt_clean_multi:,} dòng")


# ==============================================================================
# CÁCH 3: DÙNG RDD ACCUMULATOR (SINGLE-PASS RDD - DỄ HIỂU NHƯNG TỐN PYTHON SERDE)
# ==============================================================================
print("\n=== [TEST 3] CÓ DÙNG ACCUMULATOR (SINGLE-PASS RDD AUDIT) ===")
# 1. Khởi tạo Accumulator tại Driver
acc_invalid_email   = sc.accumulator(0)
acc_negative_amount = sc.accumulator(0)
acc_missing_pos     = sc.accumulator(0)
acc_clean_records   = sc.accumulator(0)

def accumulator_validate_func(row):
    global acc_invalid_email, acc_negative_amount, acc_missing_pos, acc_clean_records
    has_error = False
    
    if row.email is None or "@" not in row.email:
        acc_invalid_email.add(1)
        has_error = True
        
    if row.shipping_fee is None or row.shipping_fee < 0:
        acc_negative_amount.add(1)
        has_error = True
        
    if row.pos_code is None or len(row.pos_code.strip()) == 0:
        acc_missing_pos.add(1)
        has_error = True
        
    if not has_error:
        acc_clean_records.add(1)
        return True
    return False

t0 = time.time()
clean_rdd = df_raw.rdd.filter(accumulator_validate_func)
df_clean_acc = spark.createDataFrame(clean_rdd, df_raw.schema)

# Kích hoạt đúng 1 Action duy nhất
cnt_clean_acc = df_clean_acc.count()
t_acc = time.time() - t0

print(f"-> Hoàn thành Single-Pass Accumulator Audit (Chỉ quét 1 lần) trong: {t_acc:.3f} giây")
print(f"   + Lỗi Email (từ Acc)    : {acc_invalid_email.value:,} dòng")
print(f"   + Lỗi Phí âm (từ Acc)   : {acc_negative_amount.value:,} dòng")
print(f"   + Lỗi Thiếu POS (từ Acc) : {acc_missing_pos.value:,} dòng")
print(f"   + Dữ liệu sạch (từ Acc) : {acc_clean_records.value:,} dòng")


# ==============================================================================
# CÁCH 4: DÙNG DATAFRAME CONDITIONAL AGGREGATION (SINGLE-PASS NATIVE SPARK ENGINE)
# ==============================================================================
print("\n=== [TEST 4] DATAFRAME CONDITIONAL AGGREGATION (SINGLE-PASS NATIVE SPARK) ===")
t0 = time.time()

df_audit_native = df_raw.select(
    F.count(F.when(F.col("email").isNull() | ~F.col("email").contains("@"), 1)).alias("err_email"),
    F.count(F.when(F.col("shipping_fee").isNull() | (F.col("shipping_fee") < 0), 1)).alias("err_fee"),
    F.count(F.when(F.col("pos_code").isNull() | (F.trim(F.col("pos_code")) == ""), 1)).alias("err_pos"),
    F.count(F.when(
        F.col("email").contains("@") & \
        (F.col("shipping_fee") >= 0) & \
        F.col("pos_code").isNotNull() & \
        (F.trim(F.col("pos_code")) != ""), 1
    )).alias("clean_records")
)

audit_row = df_audit_native.collect()[0]
t_native = time.time() - t0

print(f"-> Hoàn thành Single-Pass Native DataFrame Audit trong: {t_native:.3f} giây!")
print(f"   + Lỗi Email (từ Native)    : {audit_row['err_email']:,} dòng")
print(f"   + Lỗi Phí âm (từ Native)   : {audit_row['err_fee']:,} dòng")
print(f"   + Lỗi Thiếu POS (từ Native) : {audit_row['err_pos']:,} dòng")
print(f"   + Dữ liệu sạch (từ Native) : {audit_row['clean_records']:,} dòng")


# ==============================================================================
# BẢNG TỔNG HỢP VÀ SO SÁNH HIỆU NĂNG 4 PHƯƠNG PHÁP
# ==============================================================================
print("\n=======================================================================")
print("=== BẢNG SO SÁNH KẾT QUẢ & HIỆU NĂNG 4 PHƯƠNG PHÁP AUDIT DATA QUALITY ===")
print("=======================================================================")
print(f"1. Biến Python thường : Kết quả = {py_invalid_email}  --> ❌ THẤT BẠI (Executor không gửi về được)")
print(f"2. Multi-Pass Filter  : Thời gian = {t_multi:.3f}s  --> ⚠️ Tốn kém (Quét đĩa 4 lần liên tiếp)")
print(f"3. RDD Accumulator    : Thời gian = {t_acc:.3f}s  --> 🐢 Quét 1 lần nhưng tốn Python SerDe Overhead")
print(f"4. DataFrame Native   : Thời gian = {t_native:.3f}s  --> 🚀 SIÊU TỐC (Quét 1 lần + Native JVM Engine)")
print("=======================================================================")


## PHẦN 4: SO SÁNH TRỰC DIỆN: BROADCAST VARIABLE VS BROADCAST HASH JOIN

Rất nhiều bạn nhầm lẫn giữa **`sc.broadcast(dict)`** và **`df.join(F.broadcast(df_dim), ...)`**.
Đoạn code sau đây minh họa 2 cách làm cùng một bài toán để bạn thấy rõ sự khác biệt:

In [ ]:
# BÀI TOÁN: Gắn Tên Quốc Gia từ Mã Quốc Gia cho 100,000 bản ghi

# Dữ liệu giao dịch (Fact)
df_tx = spark.range(0, 100000) \
    .withColumn("country_code", F.elt(F.col("id") % 4 + 1, F.lit("VN"), F.lit("US"), F.lit("JP"), F.lit("KR"))) \
    .withColumn("order_id", F.concat(F.lit("TX_"), F.col("id")))

# CÁCH 1: DÙNG BROADCAST HASH JOIN (BHJ) - Tầng DataFrame / SQL Engine
# Tạo 1 DataFrame nhỏ
country_df = spark.createDataFrame([
    ("VN", "Việt Nam"),
    ("US", "Hoa Kỳ"),
    ("JP", "Nhật Bản"),
    ("KR", "Hàn Quốc")
], ["code", "country_name"])

t0 = time.time()
df_result_bhj = df_tx.join(F.broadcast(country_df), df_tx.country_code == country_df.code, "left")
c1 = df_result_bhj.count()
print(f"[CÁCH 1 - BHJ]: Hoàn thành {c1:,} dòng trong {time.time() - t0:.4f}s")

# CÁCH 2: DÙNG BROADCAST VARIABLE (BV) - Tầng Python Object / Lookup Dict
country_dict = {"VN": "Việt Nam", "US": "Hoa Kỳ", "JP": "Nhật Bản", "KR": "Hàn Quốc"}
b_country = sc.broadcast(country_dict)

@F.udf(returnType=StringType())
def lookup_country(code):
    return b_country.value.get(code, "Unknown")

t0 = time.time()
df_result_bv = df_tx.withColumn("country_name", lookup_country(F.col("country_code")))
c2 = df_result_bv.count()
print(f"[CÁCH 2 - BV ]: Hoàn thành {c2:,} dòng trong {time.time() - t0:.4f}s")

print("\n=> LỜI KHUYÊN DE:")
print("- Nếu dữ liệu là 2 DataFrames: Luôn ưu tiên CÁCH 1 (Broadcast Hash Join) vì Catalyst Optimizer tối ưu bytecode C++/Java.")
print("- Nếu dữ liệu là Python dictionary/set/model: Dùng CÁCH 2 (Broadcast Variable) để tra cứu không cần tạo DataFrame.")


## PHẦN 5: CÁC CẠM BẪY CHẾT NGƯỜI (GOTCHAS) CẦN TRÁNH TRONG THỰC TẾ

| Vũ Khí | Cạm Bẫy Phổ Biến | Cách Phòng Tránh |
| :--- | :--- | :--- |
| **Broadcast Variable** | **Dữ liệu Broadcast quá lớn gây OOM:** Cố broadcast 1 object 5GB vào RAM của Executor 4GB $\rightarrow$ Executor lập tức văng lỗi Out-Of-Memory. | Chỉ broadcast dữ liệu nhỏ (tốt nhất dưới 50MB - 100MB). Nếu lớn hơn, hãy dùng Sort-Merge Join thông thường. |
| **Broadcast Variable** | **Quên giải phóng bộ nhớ (Memory Leak):** Khi xong việc, broadcast object vẫn nằm lỳ trong RAM. | Gọi `b_var.unpersist()` để giải phóng bộ nhớ đệm, hoặc `b_var.destroy()` để hủy hoàn toàn. |
| **Accumulator** | **Số đếm bị nhân đôi do Action chạy lại (Lazy Evaluation):** Nếu bạn gọi `.count()`, rồi sau đó lại gọi `.show()`, Spark sẽ chạy lại RDD 2 lần $\rightarrow$ Accumulator bị cộng gấp đôi! | Gọi `df.cache()` hoặc chỉ đọc Accumulator sau khi đã hoàn thành toàn bộ chuỗi Action. |
| **Accumulator** | **Cố đọc giá trị từ Executor:** Trong hàm worker gọi `print(acc.value)` $\rightarrow$ Bị lỗi! | Chỉ duy nhất **Driver** mới có quyền gọi `.value`. Executor chỉ có quyền `.add()`. |